In [2]:
%pip install openai

  Using cached openai-3.57.0-py3-none-any.whl.metadata (45 kB)
  Using cached jiter-0.19.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
Using cached openai-3.57.0-py3-none-any.whl (2.5 MB)
Using cached jiter-0.19.0-cp313-cp313-macosx_11_0_arm64.whl (315 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [openai]2m4/5 [openai]
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install openai-agents

  Using cached openai_agents-0.25.0-py3-none-any.whl.metadata (13 kB)
  Using cached griffelib-2.4.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached mcp-2.3.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pyjwt-2.15.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached websockets-17.4-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached mcp_types-2.3.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached sse_starlette-3.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.9.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.1 kB)
Using cached op

In [6]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [7]:
load_dotenv(override=True)

True

In [8]:
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [11]:
!ollama list

]11;?\NAME                        ID              SIZE      MODIFIED     
gemma4:latest               dc35e8d9c606    6.6 GB    24 hours ago    
llama3.2:latest             a80c4f17acd5    2.0 GB    24 hours ago    
gpt-oss:20b                 f38aa0c53da5    13 GB     7 days ago      
deepseek-coder-v2:latest    63fb193b3a9b    8.9 GB    13 days ago     
qwen2.5-coder:latest        dae161e27b0e    4.7 GB    13 days ago     
qwen3.5:27b                 64ae5b1d651f    17 GB     5 weeks ago     
deepseek-r1:1.5b            e0979632db5a    1.1 GB    3 months ago    
gpt-oss:latest              17052f91a42e    13 GB     3 months ago    
phi3:latest                 4f2222927938    2.2 GB    3 months ago    
gemma3:270m                 e7d36fb2c3b3    291 MB    3 months ago    
qwen3.6:27b                 a50eda8ed977    17 GB     3 months ago    


In [12]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
llama_3_2_ollama = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
gemma_4_ollama = OpenAIChatCompletionsModel(model="gemma4", openai_client=ollama_client)
gpt_oss_ollama = OpenAIChatCompletionsModel(model="gpt-oss", openai_client=ollama_client)


In [9]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [13]:
sales_agent1 = Agent(name="Ollama llama3.2 Sales Agent", instructions=instructions, model=llama_3_2_ollama)
sales_agent2 = Agent(name="Ollama gemma4 Sales Agent", instructions=instructions, model=gemma_4_ollama)
sales_agent3 = Agent(name="Ollama gpt-oss Sales Agent",instructions=instructions, model=gpt_oss_ollama)

In [14]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [15]:
from messenger import send_email, push

USE_EMAIL = False

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [16]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

Push: Subject: Yet another test

Hooray!


In [17]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [18]:
tools = [tool1, tool2, tool3, send_email_tool]

In [19]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [20]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

Push: Subject: Cut Your SOC 2 Audit from Weeks → Days with AI-Powered Compliance

Hi [FirstName],

I know the SOC 2 audit cycle can feel like a marathon—manual evidence collection, endless policy reviews, and the constant fear of that “unexpected” question from the auditors. What if you could turn that marathon into a sprint?

ComplAI is the first AI-driven SaaS platform that lets your team:

- Pull logs, screenshots, and policies directly from your cloud infra in real time
- Generate dynamic, audit-ready checklists that update as your environment changes
- Continuously monitor for compliance drift before it becomes an audit issue

Result: 97% of our customers report a 30-40% reduction in audit preparation time and zero surprises during final audits. 500+ orgs in SaaS, fintech, and med-tech trust ComplAI to stay on track.

Got 15 minutes next week? Let me show you a quick walkthrough and share a case study from a company very similar to yours.

Best,
Alex Martinez
Senior Solutions Engi

In [21]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [22]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [23]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [24]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [25]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [26]:
review.is_professional

False

In [27]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [28]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=llama_3_2_ollama, output_guardrails=[email_guardrail])


In [29]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

In [30]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=llama_3_2_ollama)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Howdy there, partner!

**Subject: Let's Lasso Success with AI-Powered Solutions**

Hey [First Name],

I've been ridin' these sales trails for a spell, and I've come gun-blazin' to you with a proposition that'll make your head spin. Our team at ComplAI has been chasin' ways to help businesses like yours corral the competition and drive growth like never before.

We've got a unique horse, partner - our cutting-edge Natural Language Processing (NLP) technology. It lets you tap into the vast world of customer feedback, sentiment, and intent, all in one place. That's right, with ComplAI, you'll be dealin' with the real dirt on your customers, not just guesswork or assumptions.

**Here's what we think sets us apart:**

* **AI-driven insights**: Our platform gives you real-time, actionable data on what customers are sayin' and how they're feelin'.
* **Seamless integration**: Our solutions fit snuggly with your existing workflows, so you can keep on chuggin' without a hitch.
* **Expert support

In [31]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


In [32]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [33]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [34]:
CODE_DIR

PosixPath('/Users/ming/Ming/Learning/Projects/agent-learning/code')

In [35]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [36]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [37]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [38]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [40]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

Custom tool failed


Fixed the bug in `orders.py`:
- Replaced `dict.fromkeys(..., [])` with a fresh per-key list via `setdefault`, which avoids all customers sharing the same list.
- Wrote the corrected file to `/Users/ming/Ming/Learning/Projects/agent-learning/output/orders.py`.

If you want, I can also add a tiny test for the grouping behavior.


In [41]:
from agents.mcp import MCPServerStreamableHttp

In [42]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [43]:
agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

As of my last update in October 2021, I do not have information on specific features introduced after that, including details about the SandboxAgents feature or the Manifest object in the OpenAI Agents SDK from May 2026. To get accurate and up-to-date information, I recommend checking the official OpenAI documentation or announcements regarding SandboxAgents and the Manifest object.


In [44]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

In the OpenAI Agents SDK's SandboxAgents feature, the `Manifest` object serves a crucial role in defining the workspace environment for a new sandbox session. Specifically, it outlines the structure, contents, and access rules for the sandbox where the agent operates.

### Key Functions of the Manifest Object:

1. **Defining Workspace**:
   - It specifies the `root` of the workspace and sets up the initial directory structure.

2. **Including Files and Repositories**:
   - The `Manifest` can declare specific files and directories that the agent will require, including local files, Git repositories, and external storage mounts.

3. **Permission and Path Management**:
   - It ensures that paths are workspace-relative to maintain portability across various execution environments (e.g., local, Docker, or hosted clients).

4. **Setting Environment**:
   - The `Manifest` allows you to set environment variables and define user or group access rights within the sandbox.

5. **Materializing Inp